In [ ]:
!pip install --upgrade --force-reinstall torch torchvision torchaudio triton --index-url https://download.pytorch.org/whl/cu130

Looking in indexes: https://download.pytorch.org/whl/cu130
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 67.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 51.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 69.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 98.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 121.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 169.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 89.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 223.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 156.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 91.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 68.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.9/14

In [ ]:
!pip install --upgrade transformers sentence-transformers pandas umap-learn matplotlib seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 42.1 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.4
    Uninstalling numpy-2.4.4:
      Successfully uninstalled numpy-2.4.4
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: matplotlib
    Found existing installat

In [ ]:
import huggingface_hub

huggingface_hub.login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import pandas as pd
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import ndcg_score

# 1. Load the model
model = SentenceTransformer("BAAI/bge-m3", trust_remote_code=True)

if torch.cuda.is_available():
    model = model.to('cuda')

# 2. Load data
df_vectors = pd.read_csv('result-qwen3_0_6b.csv')

# Identify all embedding columns (embedding_0, embedding_1, ...)
embedding_cols = sorted([col for col in df_vectors.columns if col.startswith('embedding_')], key=lambda x: int(x.split('_')[1]))

# Drop rows with NaN in the embedding columns
df_final = df_vectors.dropna(subset=embedding_cols).reset_index(drop=True)

# Merge all embedding columns into a single numpy array
doc_embeddings = df_final[embedding_cols].values.astype('float32')

print(f"Successfully loaded {len(df_final)} rows from result-qwen3_0_6b.csv.")
print(f"Embedding columns merged: {len(embedding_cols)} dimensions.")
print(f"Final doc_embeddings shape: {doc_embeddings.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Successfully loaded 11384 rows from result-qwen3_0_6b.csv.
Embedding columns merged: 1024 dimensions.
Final doc_embeddings shape: (11384, 1024)


In [ ]:
def evaluate_ndcg(query, ground_truth_ids, k=10):
    # Encode query

    query_embedding = model.encode([f"Search Recommend by Query: {query}"])

    # Calculate cosine similarity
    similarities = cosine_similarity(query_embedding, doc_embeddings)[0]

    # Map relevance using 'movie_id' from df_final
    relevance = df_final['movie_id'].apply(lambda x: 1 if x in ground_truth_ids else 0).values

    if np.sum(relevance) == 0:
        print(f"Warning: No relevant documents found in dataset for query: {query}")
        return 0.0

    score = ndcg_score([relevance], [similarities], k=k)
    return score

# Define test set based on df_final metadata (genres column)
action_movie_ids = df_final[df_final['genres'].str.contains('액션', na=False)]['movie_id'].unique().tolist()

test_queries = [
    {"query": "화끈하고 박진감 넘치는 액션 영화", "relevant_ids": action_movie_ids}
]

results = []
for item in test_queries:
    s = evaluate_ndcg(item['query'], item['relevant_ids'])
    results.append(s)
    print(f"Query: {item['query']} | NDCG@10: {s:.4f}")

if results:
    print(f"\nAverage NDCG@10: {np.mean(results):.4f}")

Query: 화끈하고 박진감 넘치는 액션 영화 | NDCG@10: 0.9266

Average NDCG@10: 0.9266


In [ ]:
import random
import numpy as np
from sklearn.metrics import ndcg_score, average_precision_score
from sklearn.metrics.pairwise import cosine_similarity

# 1. Define query generation patterns
unique_genres = set()
for g in df_final['genres'].dropna().unique():
    cleaned = g.replace("'", "").split(',')
    unique_genres.update([c.strip() for c in cleaned])

# 2. Updated Evaluation function for multiple metrics
def evaluate_batch_2(queries, k_list=[10, 50], filter=False):
    # Metrics to track
    results = {k: {'ndcg': [], 'precision': [], 'recall': [], 'ap': []} for k in k_list}

    for idx, item in enumerate(queries):
        query_embedding = model.encode([f"Search Recommend by Query: {item['query']}"])

        # Filter logic: only include relevant_ids in the pool
        if filter:
            mask = df_final['movie_id'].isin(item['relevant_ids']).values
            eff_embeddings = doc_embeddings[mask]
            eff_df = df_final[mask].reset_index(drop=True)
        else:
            eff_embeddings = doc_embeddings
            eff_df = df_final

        if len(eff_embeddings) == 0: continue

        similarities = cosine_similarity(query_embedding, eff_embeddings)[0]
        y_true = eff_df['movie_id'].apply(lambda x: 1 if x in item['relevant_ids'] else 0).values
        total_relevant = np.sum(y_true)

        if total_relevant == 0: continue

        sorted_indices = np.argsort(similarities)[::-1]

        for k in k_list:
            current_k = min(k, len(y_true))
            top_k_indices = sorted_indices[:current_k]

            num_relevant_at_k = np.sum(y_true[top_k_indices])
            precision = num_relevant_at_k / current_k
            recall = num_relevant_at_k / total_relevant

            # Handle NDCG error for single document cases
            if len(y_true) > 1:
                ndcg = ndcg_score([y_true], [similarities], k=current_k)
            else:
                # If only 1 doc exists and it's relevant, NDCG is 1.0
                ndcg = 1.0 if y_true[0] == 1 else 0.0

            ap = average_precision_score(y_true[top_k_indices], similarities[top_k_indices]) if num_relevant_at_k > 0 else 0.0

            results[k]['ndcg'].append(ndcg)
            results[k]['precision'].append(precision)
            results[k]['recall'].append(recall)
            results[k]['ap'].append(ap)

        if (idx + 1) % 20 == 0:
            print(f"Progress: {idx+1}/{len(queries)} processed...")

    final_metrics = {}
    for k in k_list:
        for m in ['ndcg', 'precision', 'recall', 'ap']:
            final_metrics[f"{m.upper()}@{k}"] = np.mean(results[k][m]) if results[k][m] else 0.0

    return final_metrics

In [ ]:
import pandas as pd
import numpy as np
import random
import re

# Helper for relevance mapping
def get_relevant_ids(df, column, target_value):
    safe_value = re.escape(target_value)
    return df[df[column].str.contains(safe_value, na=False, case=False)]['movie_id'].unique().tolist()

# Helper to run eval per category with specific query count
def run_evaluation_for_category(category_name, num_queries, filter):
    all_values = set()
    for entry in df_final[category_name].dropna().unique():
        parts = [p.strip().replace("'", "") for p in str(entry).split(',')]
        all_values.update(parts)

    val_list = [v for v in all_values if len(v) > 1]
    if not val_list: return None

    queries = []
    templates = ["{} 영화", "{} 관련 추천", "{} 테마의 작품", "{} 느낌", "{} 장르의 명작", "{} 분위기 추천"]

    # Sampling with replacement to ensure exactly num_queries
    selected_vals = random.choices(val_list, k=num_queries)

    for val in selected_vals:
        q_text = random.choice(templates).format(val)
        rel_ids = get_relevant_ids(df_final, category_name, val)
        if rel_ids:
            queries.append({"query": q_text, "relevant_ids": rel_ids})

    if not queries: return None
    return evaluate_batch_2(queries, k_list=[10, 50], filter=filter)

# 1. Configuration for development report
categories = ['genres', 'moods', 'themes']
sample_sizes = [{ 'size': 200, 'filter': False}, {'size': 200, 'filter': True}]
report_rows = []

for size in sample_sizes:
    print(f"\n--- Running Evaluation with Fixed Query Size: {size['size']} ---")

    # Category Evals
    for cat in categories:
        print(f"Processing Category: {cat}...")
        scores = run_evaluation_for_category(cat, size['size'], size['filter'])
        if scores:
            scores['Category'] = cat
            scores['Query_Count'] = size['size']
            scores['Filter'] = size['filter']
            report_rows.append(scores)

    # Complex Query Eval for this size
    print(f"Processing Complex Queries...")
    complex_queries = []
    for _ in range(size['size']):
        g = random.choice(list(unique_genres))
        m_entry = df_final['moods'].dropna().sample(1).iloc[0]
        m = random.choice(m_entry.replace("'", "").split(','))
        q_text = f"{m.strip()} 분위기의 {g.strip()} 영화"
        rel_ids = df_final[(df_final['genres'].str.contains(re.escape(g.strip()), na=False)) &
                           (df_final['moods'].str.contains(re.escape(m.strip()), na=False))]['movie_id'].unique().tolist()
        if rel_ids:
            complex_queries.append({"query": q_text, "relevant_ids": rel_ids})

    if complex_queries:
        c_scores = evaluate_batch_2(complex_queries, k_list=[10, 50], filter=size['filter'])
        c_scores['Category'] = 'Complex (Genre+Mood)'
        c_scores['Query_Count'] = size['size']
        c_scores['Filter'] = size['filter']
        report_rows.append(c_scores)

# 2. Format Final Development Report
df_report = pd.DataFrame(report_rows)

# Organize columns
main_cols = ['Category', 'Query_Count', 'Filter']
metric_cols = sorted([c for c in df_report.columns if '@' in c], key=lambda x: (x.split('@')[0], int(x.split('@')[1])))
df_report = df_report[main_cols + metric_cols]

print("\n[Development Performance Report: Fixed Query Counts]")
display(df_report.style.format(precision=4).background_gradient(cmap='YlGnBu', subset=metric_cols))


--- Running Evaluation with Fixed Query Size: 200 ---
Processing Category: genres...
Progress: 20/200 processed...
Progress: 40/200 processed...
Progress: 60/200 processed...
Progress: 80/200 processed...
Progress: 100/200 processed...
Progress: 120/200 processed...
Progress: 140/200 processed...
Progress: 160/200 processed...
Progress: 180/200 processed...
Progress: 200/200 processed...
Processing Category: moods...
Progress: 20/200 processed...
Progress: 40/200 processed...
Progress: 60/200 processed...
Progress: 80/200 processed...
Progress: 100/200 processed...
Progress: 120/200 processed...
Progress: 140/200 processed...
Progress: 160/200 processed...
Progress: 180/200 processed...
Progress: 200/200 processed...
Processing Category: themes...
Progress: 20/200 processed...
Progress: 40/200 processed...
Progress: 60/200 processed...
Progress: 80/200 processed...
Progress: 100/200 processed...
Progress: 120/200 processed...
Progress: 140/200 processed...
Progress: 160/200 processed.

,Category,Query_Count,Filter,AP@10,AP@50,NDCG@10,NDCG@50,PRECISION@10,PRECISION@50,RECALL@10,RECALL@50
0,genres,200,False,0.4325,0.3820,0.3660,0.3357,0.3400,0.2877,0.0512,0.1312
1,moods,200,False,0.4118,0.3674,0.3226,0.3138,0.3205,0.3106,0.0034,0.0172
2,themes,200,False,0.6159,0.5524,0.5368,0.5027,0.5295,0.4918,0.0057,0.0228
3,Complex (Genre+Mood),200,False,0.2411,0.1895,0.1500,0.1584,0.1342,0.1226,0.0323,0.1026
4,genres,200,True,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.3145,0.5971
5,moods,200,True,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0337,0.1656
6,themes,200,True,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0161,0.0803
7,Complex (Genre+Mood),200,True,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.4782,0.7254
